# Summary

01_quickstart.ipynb

Single scan inference end-to-end. Load a scan, run NeuroFM.predict(), show the output brain health and latent features.

# Install

Unfortunately due to dependency conflicts between Tensorflow 2.13 and ipython, installation needs to take place outside the notebook in your ipykernel environment (unless you're running a kernel from the NeuroFM docker container, then you can skip this step).

Please run the following in your terminal Python environment before starting the kernel (mamba, etc.) to create and activate a dedicated environment, then install dependencies:
```bash
pip install "neurofm[notebooks] @ git+https://github.com/rockNroll87q/NeuroFM.git@preprint-prep"
pip install templateflow==25.1.2 "numpy<=1.24.3" "typing-extensions<4.6.0"
```

**Note:** The `typing-extensions` pin is required for TensorFlow compatibility. Installing inside the notebook is not recommended as it conflicts with ipython's own dependencies. 

If you run into issues, we recommend using the NeuroFM notebook docker container.

# Python API

## Imports

In [ ]:
# Initial import may take ~30s due to internal TensorFlow import.
import templateflow.api as tflow
from neurofm import NeuroFM
from neurofm.model import BRAIN_HEALTH_KEYS

## Create template test vol

For the purposes of illustration, we load an MNI test volume. In your own use-case, this can be replaced by a single volume, a directory of volumes, or a `.csv` file with an input column pointing to your volume files.

In [2]:
print("Fetching MNI152NLin2009cAsym 1mm T1w template from TemplateFlow...")
template_path = tflow.get(
    "MNI152NLin2009cAsym",
    resolution=1,
    desc="brain",
    suffix="T1w",
    extension=".nii.gz",
)

if template_path is None:
    print("TemplateFlow could not fetch the template. Check your internet connection.")

Fetching MNI152NLin2009cAsym 1mm T1w template from TemplateFlow...


## Run NeuroFM

In [3]:
model_variant = "neurofm-s" # choose your variant
device = "cpu"

In [ ]:
model = NeuroFM(
    variant=model_variant,
    device=device,
)

In [5]:
results = model.predict(template_path, outputs=['brain_health', 'latent'])
brain_health = results['brain_health']
latent = results['latent']

2026-03-18 13:18:52.091 | DEBUG    | neurofm.io:_reorient:161 - Reorienting from 'RAS' to 'LIA'.
2026-03-18 13:18:52.139 | DEBUG    | neurofm.io:_resample:184 - Resampling: shape (193, 193, 229) -> (256, 256, 256), zooms (1.0, 1.0, 1.0) -> (1.0, 1.0, 1.0).


### Brain health outputs

In [6]:
print(f'Output order: {BRAIN_HEALTH_KEYS}')

Output order: ['brain_age', 'sex', 'ventricle_volume', 'brain_volume']


In [7]:
print(f'Predicted age: {brain_health[0]} years')
print(f'Predicted sex: {brain_health[1]} (0 == F, 1 == M)')
print(f'Predicted ventricle vol: {brain_health[2]} mm^3') 
print(f'Predicted brain vol: {brain_health[3]} mm^3')

Predicted age: 58.32535934448242 years
Predicted sex: 0.0 (0 == F, 1 == M)
Predicted ventricle vol: 23985.23828125 mm^3
Predicted brain vol: 1695925.25 mm^3


### Latent outputs

Latent embedded representation of the input brain volume. Dimensionality depends on model variant.

In [8]:
print(latent)

[ 0.03749801  0.32106072 -0.18453547 -0.20148537  0.1072489  -0.21201581
  0.08957285  0.41148937  0.43628114 -0.13389575 -0.16290452 -0.03286824
  0.12378883  0.27158815 -0.00743028 -0.08271645  0.16566369  0.2677508
  0.08607052 -0.21734777  0.04916117  0.02662336 -0.09265818  0.0063838
  0.3437472  -0.08816005  0.24651705 -0.16467224 -0.124823    0.13885532
  0.49529997  0.29678926  0.33648756  0.0842379   0.06424014  0.43970048
  0.4043522   0.36315382  0.09246086  0.09763109 -0.04595786  0.357244
 -0.17047833  0.14726664  0.2202377   0.42131206  0.4830866   0.35445255
  0.29830086  0.19760221  0.04977968  0.09764886  0.17741829  0.47973597
 -0.19878125  0.44830596 -0.05329991  0.43187457 -0.12600078  0.1516798
  0.24960637 -0.18141511  0.33492997 -0.00737659  0.11763795  0.25770366
 -0.11370428  0.27564722  0.3271057   0.15555403  0.265876    0.18161544
  0.38085103  0.3060128   0.00314285 -0.10060848  0.16663937  0.19807285
  0.320735    0.09081508  0.355959   -0.17386442  0.1064

# Script method

## Create template test vol

Create the template test data and save to /tmp/

In [ ]:
!python ../scripts/get_test_data.py -o /tmp/test_data/ --mode template -n 1

## Run inference script to produce output files

In [ ]:
!python ../scripts/run_inference.py -i /tmp/test_data/MNI152_test.nii.gz \
    -o /tmp/nfm_results --device cpu --model neurofm-s --outputs brain_health,latent

In [13]:
!ls /tmp/nfm_results

individuals                 latent_embeddings_index.csv
latent_embeddings.npy       results_summary.csv


Show the output summary .csv (brain health data for all volumes)

In [14]:
import pandas as pd

pd.read_csv('/tmp/nfm_results/results_summary.csv')

,input,brain_age,sex,ventricle_volume,brain_volume
0,/tmp/test_data/MNI152_test.nii.gz,58.325359,0.0,23985.238281,1695925.25
